# Tutorial: Ejercicio 3.3 en Python - inventario de mascarillas super for dummies

Audiencia:
- Personas que recien estan empezando en optimizacion.
- Personas que quieren entender el problema antes de ver AMPL.

Prerrequisitos:
- Saber leer listas, diccionarios y funciones muy simples en Python.
- Entender que inventario significa "lo que sobra y se guarda".

Objetivos:
- Entender que decide el modelo.
- Revisar si un plan es factible o no.
- Encontrar el plan mas barato con una busqueda pequena y facil de seguir.


## Enunciado del ejercicio

Una empresa distribuidora de tapabocas tiene demanda pronosticada para los proximos cuatro meses de `2800`, `2200`, `3200` y `2500` unidades para los meses `1`, `2`, `3` y `4`, respectivamente.

Datos del enunciado:
- capacidad de produccion normal: `2700` unidades por mes,
- capacidad adicional en tiempo extra: `300` unidades por mes,
- sobrecosto de tiempo extra: `10` USD por unidad,
- costo de almacenamiento: `2` USD por unidad que quede guardada al final de un mes.

Se asume inventario inicial `0` y se desea inventario final `0`.

Pregunta:
**¿Cual es el programa optimo de produccion que minimiza los costos de tiempo extra y almacenamiento, cumpliendo la demanda de cada mes?**


## Mapa del notebook

1. Ver los datos del problema.
2. Entender que decisiones existen.
3. Usar la ecuacion mas importante del inventario.
4. Revisar un plan completo.
5. Buscar el mejor plan.
6. Probar una variante como ejercicio.


In [19]:
MESES = [1, 2, 3, 4]
DEMANDA = {1: 2800, 2: 2200, 3: 3200, 4: 2500}
CAP_NORMAL = 2700
CAP_EXTRA = 300
COSTO_EXTRA = 10
COSTO_INVENTARIO = 2
INV_MAX = {1: 200, 2: 800, 3: 300, 4: 0}

resumen_datos = {
    "meses": MESES,
    "demanda": DEMANDA,
    "capacidad_normal_por_mes": CAP_NORMAL,
    "capacidad_extra_por_mes": CAP_EXTRA,
    "costo_por_unidad_extra": COSTO_EXTRA,
    "costo_por_unidad_guardada": COSTO_INVENTARIO,
}

resumen_datos


{'meses': [1, 2, 3, 4],
 'demanda': {1: 2800, 2: 2200, 3: 3200, 4: 2500},
 'capacidad_normal_por_mes': 2700,
 'capacidad_extra_por_mes': 300,
 'costo_por_unidad_extra': 10,
 'costo_por_unidad_guardada': 2}

## Paso 0 - Que esta pasando aqui

La empresa debe cubrir la demanda de mascarillas durante 4 meses.

Cada mes puede hacer tres cosas importantes:
- producir en horario normal,
- producir extra si falta capacidad,
- guardar inventario para usarlo despues.

Como producir extra cuesta y guardar inventario tambien cuesta, queremos responder esta pregunta:

**Cual es el plan que cumple todo y cuesta menos?**


In [20]:
tabla_meses = []
for mes in MESES:
    tabla_meses.append(
        {
            "mes": mes,
            "demanda": DEMANDA[mes],
            "normal_max": CAP_NORMAL,
            "extra_max": CAP_EXTRA,
            "inventario_max_fin_de_mes": INV_MAX[mes],
        }
    )

tabla_meses


[{'mes': 1,
  'demanda': 2800,
  'normal_max': 2700,
  'extra_max': 300,
  'inventario_max_fin_de_mes': 200},
 {'mes': 2,
  'demanda': 2200,
  'normal_max': 2700,
  'extra_max': 300,
  'inventario_max_fin_de_mes': 800},
 {'mes': 3,
  'demanda': 3200,
  'normal_max': 2700,
  'extra_max': 300,
  'inventario_max_fin_de_mes': 300},
 {'mes': 4,
  'demanda': 2500,
  'normal_max': 2700,
  'extra_max': 300,
  'inventario_max_fin_de_mes': 0}]

## Paso 1 - Que decide el modelo

Para no perdernos, separamos las decisiones en dos niveles.

Decisiones completas del modelo:
- `produccion_normal[mes]`
- `produccion_extra[mes]`
- `inventario[mes]`

Decision pedagogica de este notebook:
- si sabemos el inventario al final de cada mes, la produccion sale sola.

Todavia mejor:
- como en el mes 4 la demanda es `2500` y la capacidad normal es `2700`, no hace falta llegar con stock al mes 4,
- por eso podemos fijar `inventario[3] = 0` e `inventario[4] = 0`,
- entonces solo probamos `inventario[1]` e `inventario[2]`.

Esa es la razon por la que aqui la busqueda queda muy pequena.


In [21]:
variables_del_modelo = [
    {
        "variable": "produccion_normal[mes]",
        "significado": "cuanto producimos en horario normal en ese mes",
    },
    {
        "variable": "produccion_extra[mes]",
        "significado": "cuanto producimos extra en ese mes",
    },
    {
        "variable": "inventario[mes]",
        "significado": "cuanto dejamos guardado al final de ese mes",
    },
    {
        "variable": "inventario[0]",
        "significado": "inventario inicial; en este problema vale 0",
    },
]

variables_del_modelo


[{'variable': 'produccion_normal[mes]',
  'significado': 'cuanto producimos en horario normal en ese mes'},
 {'variable': 'produccion_extra[mes]',
  'significado': 'cuanto producimos extra en ese mes'},
 {'variable': 'inventario[mes]',
  'significado': 'cuanto dejamos guardado al final de ese mes'},
 {'variable': 'inventario[0]',
  'significado': 'inventario inicial; en este problema vale 0'}]

## Paso 2 - La ecuacion mas importante

La ecuacion clave es esta:

`produccion del mes = demanda del mes + inventario final - inventario inicial`

En palabras:
- partes con lo que ya tenias guardado,
- produces lo que haga falta,
- atiendes la demanda,
- dejas el resto guardado.

Si entiendes esta ecuacion, entiendes el corazon del problema.


In [22]:
def produccion_necesaria(demanda_mes, inventario_inicial, inventario_final):
    return demanda_mes + inventario_final - inventario_inicial


ejemplos = []
for inventario_final in [0, 100, 200]:
    produccion = produccion_necesaria(DEMANDA[1], 0, inventario_final)
    ejemplos.append(
        {
            "mes": 1,
            "inventario_inicial": 0,
            "inventario_final": inventario_final,
            "produccion_necesaria": produccion,
            "necesita_extra": produccion > CAP_NORMAL,
        }
    )

ejemplos


[{'mes': 1,
  'inventario_inicial': 0,
  'inventario_final': 0,
  'produccion_necesaria': 2800,
  'necesita_extra': True},
 {'mes': 1,
  'inventario_inicial': 0,
  'inventario_final': 100,
  'produccion_necesaria': 2900,
  'necesita_extra': True},
 {'mes': 1,
  'inventario_inicial': 0,
  'inventario_final': 200,
  'produccion_necesaria': 3000,
  'necesita_extra': True}]

## Paso 3 - Revisar un mes sin complicarnos

En un solo mes, primero calculamos cuanta produccion total hace falta.
Despues la separamos en:
- produccion normal,
- produccion extra.

Si se pasa de la capacidad total, el plan no sirve.


In [23]:
def revisar_mes(mes, inventario_inicial, inventario_final, cap_normal=CAP_NORMAL, cap_extra=CAP_EXTRA):
    produccion_total = produccion_necesaria(DEMANDA[mes], inventario_inicial, inventario_final)

    if produccion_total < 0:
        return {"factible": False, "motivo": "la produccion no puede ser negativa"}

    if produccion_total > cap_normal + cap_extra:
        return {"factible": False, "motivo": "la produccion supera la capacidad total"}

    produccion_extra = max(0, produccion_total - cap_normal)
    produccion_normal = produccion_total - produccion_extra

    return {
        "factible": True,
        "produccion_total": produccion_total,
        "produccion_normal": produccion_normal,
        "produccion_extra": produccion_extra,
    }


revisar_mes(mes=1, inventario_inicial=0, inventario_final=0)


{'factible': True,
 'produccion_total': 2800,
 'produccion_normal': 2700,
 'produccion_extra': 100}

## Paso 4 - Revisar un plan completo

Un plan completo dice cuanto inventario queda al final de cada mes.
Con eso podemos revisar mes por mes si el plan:
- cumple la demanda,
- respeta las capacidades,
- y cuanto cuesta.


In [24]:
def evaluar_plan(inventario, cap_normal=CAP_NORMAL, cap_extra=CAP_EXTRA, costo_extra=COSTO_EXTRA, costo_inventario=COSTO_INVENTARIO):
    detalle = []
    produccion_normal = {}
    produccion_extra = {}
    costo_total = 0

    for mes in MESES:
        revision = revisar_mes(
            mes=mes,
            inventario_inicial=inventario[mes - 1],
            inventario_final=inventario[mes],
            cap_normal=cap_normal,
            cap_extra=cap_extra,
        )

        if not revision["factible"]:
            return None

        produccion_normal[mes] = revision["produccion_normal"]
        produccion_extra[mes] = revision["produccion_extra"]

        costo_mes = costo_extra * revision["produccion_extra"] + costo_inventario * inventario[mes]
        costo_total += costo_mes

        detalle.append(
            {
                "mes": mes,
                "normal": revision["produccion_normal"],
                "extra": revision["produccion_extra"],
                "inventario_final": inventario[mes],
                "costo_mes": costo_mes,
            }
        )

    return {
        "normal": produccion_normal,
        "extra": produccion_extra,
        "inventario": inventario,
        "detalle": detalle,
        "costo": costo_total,
    }


plan_del_libro = {0: 0, 1: 0, 2: 500, 3: 0, 4: 0}
evaluacion_libro = evaluar_plan(plan_del_libro)
evaluacion_libro


{'normal': {1: 2700, 2: 2700, 3: 2700, 4: 2500},
 'extra': {1: 100, 2: 0, 3: 0, 4: 0},
 'inventario': {0: 0, 1: 0, 2: 500, 3: 0, 4: 0},
 'detalle': [{'mes': 1,
   'normal': 2700,
   'extra': 100,
   'inventario_final': 0,
   'costo_mes': 1000},
  {'mes': 2,
   'normal': 2700,
   'extra': 0,
   'inventario_final': 500,
   'costo_mes': 1000},
  {'mes': 3,
   'normal': 2700,
   'extra': 0,
   'inventario_final': 0,
   'costo_mes': 0},
  {'mes': 4,
   'normal': 2500,
   'extra': 0,
   'inventario_final': 0,
   'costo_mes': 0}],
 'costo': 2000}

## Paso 5 - Buscar el mejor plan

Ahora si hacemos optimizacion, pero en version amable.

Como solo probamos `inventario[1]` e `inventario[2]`, podemos revisar todos los casos posibles:
- desde `inventario[1] = 0` hasta `200`,
- y desde `inventario[2] = 0` hasta `800`.

Despues nos quedamos con el plan factible de menor costo.


In [25]:
def buscar_mejor_plan(cap_normal=CAP_NORMAL, cap_extra=CAP_EXTRA, costo_extra=COSTO_EXTRA, costo_inventario=COSTO_INVENTARIO):
    mejor_plan = None
    planes_probados = 0
    planes_factibles = 0

    for inventario_1 in range(INV_MAX[1] + 1):
        for inventario_2 in range(INV_MAX[2] + 1):
            planes_probados += 1
            inventario = {0: 0, 1: inventario_1, 2: inventario_2, 3: 0, 4: 0}
            plan = evaluar_plan(
                inventario,
                cap_normal=cap_normal,
                cap_extra=cap_extra,
                costo_extra=costo_extra,
                costo_inventario=costo_inventario,
            )

            if plan is None:
                continue

            planes_factibles += 1

            if mejor_plan is None or plan["costo"] < mejor_plan["costo"]:
                mejor_plan = plan

    return mejor_plan, planes_probados, planes_factibles


mejor_plan, planes_probados, planes_factibles = buscar_mejor_plan()

print("Planes probados:", planes_probados)
print("Planes factibles:", planes_factibles)
print("Costo minimo encontrado:", mejor_plan["costo"])
print("Inventario optimo:", mejor_plan["inventario"])

assert mejor_plan["costo"] == 2000
assert mejor_plan["inventario"] == {0: 0, 1: 0, 2: 500, 3: 0, 4: 0}

mejor_plan["detalle"]


Planes probados: 161001
Planes factibles: 120801
Costo minimo encontrado: 2000
Inventario optimo: {0: 0, 1: 0, 2: 500, 3: 0, 4: 0}


[{'mes': 1,
  'normal': 2700,
  'extra': 100,
  'inventario_final': 0,
  'costo_mes': 1000},
 {'mes': 2,
  'normal': 2700,
  'extra': 0,
  'inventario_final': 500,
  'costo_mes': 1000},
 {'mes': 3, 'normal': 2700, 'extra': 0, 'inventario_final': 0, 'costo_mes': 0},
 {'mes': 4, 'normal': 2500, 'extra': 0, 'inventario_final': 0, 'costo_mes': 0}]

## Paso 6 - Leer la solucion como persona normal

La solucion optima dice algo muy razonable:
- en el mes 1 se produce un poco extra,
- en el mes 2 se aprovecha la capacidad normal para fabricar de mas y guardar `500`,
- en el mes 3 ese inventario guardado ayuda a cubrir la demanda alta,
- en el mes 4 se termina sin stock.

O sea: el modelo no esta haciendo magia. Solo esta adelantando produccion cuando conviene.


In [26]:
interpretacion = []
for fila in mejor_plan["detalle"]:
    interpretacion.append(
        {
            "mes": fila["mes"],
            "mensaje": (
                f"Mes {fila['mes']}: normal={fila['normal']}, extra={fila['extra']}, "
                f"inventario final={fila['inventario_final']}, costo del mes={fila['costo_mes']}"
            ),
        }
    )

interpretacion


[{'mes': 1,
  'mensaje': 'Mes 1: normal=2700, extra=100, inventario final=0, costo del mes=1000'},
 {'mes': 2,
  'mensaje': 'Mes 2: normal=2700, extra=0, inventario final=500, costo del mes=1000'},
 {'mes': 3,
  'mensaje': 'Mes 3: normal=2700, extra=0, inventario final=0, costo del mes=0'},
 {'mes': 4,
  'mensaje': 'Mes 4: normal=2500, extra=0, inventario final=0, costo del mes=0'}]

## Ejercicio guiado

Antes de correr la siguiente celda, intenta adivinar:
- que pasa si la capacidad normal sube de `2700` a `2800`,
- si el costo baja,
- y si todavia hace falta producir extra.

Error comun:
- poner mal la ecuacion del inventario y cambiar el signo de `inventario_final`.


In [27]:
escenario_mas_facil, _, _ = buscar_mejor_plan(cap_normal=2800)

{
    "costo_original": mejor_plan["costo"],
    "costo_con_capacidad_normal_2800": escenario_mas_facil["costo"],
    "extra_con_capacidad_normal_2800": escenario_mas_facil["extra"],
    "inventario_con_capacidad_normal_2800": escenario_mas_facil["inventario"],
}


{'costo_original': 2000,
 'costo_con_capacidad_normal_2800': 800,
 'extra_con_capacidad_normal_2800': {1: 0, 2: 0, 3: 0, 4: 0},
 'inventario_con_capacidad_normal_2800': {0: 0, 1: 0, 2: 400, 3: 0, 4: 0}}